# 🔍 Anomaly Detection: Mobile Money Fraud in East Africa

**Author:** Jok Akech Atem Mabior | CMU Africa  
**Dataset:** Synthetic M-Pesa style transaction data  
**Goal:** Detect fraudulent mobile money transactions using unsupervised anomaly detection techniques.

## Background

Mobile money has transformed financial inclusion in East Africa — Kenya's **M-Pesa** alone processes over $300 billion annually, while **MTN Mobile Money**, **Airtel Money**, and **Equitel** serve millions across Uganda, Tanzania, Rwanda, and Ethiopia.

However, this rapid growth has attracted sophisticated fraud. Common fraud types include:

| Fraud Type | Description |
|---|---|
| **SIM Swap Fraud** | Criminal convinces telecom to transfer victim's number to their SIM |
| **Agent Fraud** | Rogue M-Pesa agents manipulate transactions or steal credentials |
| **Phishing/Vishing** | Social engineering to obtain PINs via fake SMS or phone calls |
| **Money Laundering** | Structuring transactions to avoid detection thresholds |
| **Account Takeover** | Unauthorized access to accounts via stolen credentials |

Financial losses from mobile money fraud in sub-Saharan Africa are estimated at **$4 billion annually** (ITU, 2023). Machine learning-based anomaly detection can flag suspicious transactions in **real-time**, protecting millions of unbanked users who have no other safety net.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.svm import OneClassSVM
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import PCA
from sklearn.metrics import (classification_report, confusion_matrix,
                              roc_auc_score, precision_recall_curve,
                              average_precision_score, roc_curve)
import warnings
warnings.filterwarnings('ignore')
np.random.seed(42)
plt.style.use('seaborn-v0_8')
print("Libraries loaded!")

## 1. Data Generation

We simulate a mobile money transaction dataset with realistic East African patterns. Normal transactions follow log-normal distributions reflecting typical M-Pesa usage. Fraudulent transactions exhibit anomalous patterns: unusually large amounts, unusual hours, high frequency, new recipients, and accounts with more failed PIN attempts.

In [ ]:
np.random.seed(42)
N_NORMAL = 4750
N_FRAUD  = 250

TRANSACTION_TYPES = ['Send Money', 'Buy Airtime', 'Pay Bill', 'Withdraw', 'Deposit', 'Buy Goods']
AGENT_REGIONS = ['Nairobi', 'Mombasa', 'Kisumu', 'Nakuru', 'Kampala', 'Dar es Salaam', 'Kigali']

def gen_normal_transactions(n):
    hour_probs = [0.005]*5 + [0.01]*3 + [0.05]*2 + [0.08]*4 + [0.07]*4 + [0.05]*4 + [0.02]*2
    return pd.DataFrame({
        'amount_ksh': np.random.lognormal(8.5, 1.2, n).clip(10, 70000),
        'transaction_type': np.random.choice(TRANSACTION_TYPES, n, p=[0.35, 0.20, 0.15, 0.15, 0.10, 0.05]),
        'hour_of_day': np.random.choice(range(24), n, p=hour_probs),
        'day_of_week': np.random.choice(range(7), n),
        'agent_region': np.random.choice(AGENT_REGIONS, n),
        'account_age_days': np.random.normal(700, 300, n).clip(30, 3000),
        'prev_7day_txn_count': np.random.poisson(8, n).clip(0, 50).astype(float),
        'prev_7day_volume_ksh': np.random.lognormal(9.0, 1.0, n).clip(100, 500000),
        'recipient_age_days': np.random.normal(600, 250, n).clip(1, 3000),
        'failed_pin_attempts': np.random.choice([0, 1, 2], n, p=[0.92, 0.06, 0.02]).astype(float),
        'distance_from_home_km': np.random.exponential(5, n).clip(0, 50),
        'is_new_recipient': np.random.binomial(1, 0.25, n).astype(float),
        'fraud': 0
    })

def gen_fraud_transactions(n):
    fraud_hour_probs = [0.10]*5 + [0.04]*4 + [0.02]*5 + [0.02]*5 + [0.02]*5
    return pd.DataFrame({
        'amount_ksh': np.random.lognormal(10.5, 1.5, n).clip(5000, 500000),
        'transaction_type': np.random.choice(TRANSACTION_TYPES, n, p=[0.50, 0.05, 0.05, 0.25, 0.10, 0.05]),
        'hour_of_day': np.random.choice(range(24), n, p=fraud_hour_probs),
        'day_of_week': np.random.choice(range(7), n),
        'agent_region': np.random.choice(AGENT_REGIONS, n),
        'account_age_days': np.random.normal(200, 150, n).clip(1, 500),
        'prev_7day_txn_count': np.random.poisson(30, n).clip(10, 100).astype(float),
        'prev_7day_volume_ksh': np.random.lognormal(12.0, 1.0, n).clip(10000, 2000000),
        'recipient_age_days': np.random.normal(50, 40, n).clip(1, 200),
        'failed_pin_attempts': np.random.choice([0, 1, 2, 3], n, p=[0.4, 0.2, 0.2, 0.2]).astype(float),
        'distance_from_home_km': np.random.exponential(30, n).clip(5, 500),
        'is_new_recipient': np.random.binomial(1, 0.85, n).astype(float),
        'fraud': 1
    })

normal_df = gen_normal_transactions(N_NORMAL)
fraud_df  = gen_fraud_transactions(N_FRAUD)
df = pd.concat([normal_df, fraud_df], ignore_index=True)
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

print(f"Dataset: {df.shape}")
print(f"Fraud rate: {df['fraud'].mean()*100:.1f}%")
print(f"Fraud transactions: {df['fraud'].sum()}")
df.describe().round(2)

## 2. Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 11))

features_to_compare = ['amount_ksh', 'hour_of_day', 'account_age_days',
                        'prev_7day_txn_count', 'distance_from_home_km', 'failed_pin_attempts']

for i, feat in enumerate(features_to_compare):
    ax = axes[i // 3][i % 3]
    normal_vals = df[df['fraud'] == 0][feat]
    fraud_vals  = df[df['fraud'] == 1][feat]
    ax.hist(normal_vals, bins=30, alpha=0.6, color='steelblue', label='Normal', density=True, edgecolor='none')
    ax.hist(fraud_vals,  bins=30, alpha=0.7, color='red',       label='Fraud',  density=True, edgecolor='none')
    ax.set_title(f'{feat}', fontweight='bold', fontsize=10)
    ax.set_xlabel(feat, fontsize=9)
    ax.legend(fontsize=8)
plt.suptitle('Feature Distributions: Fraud vs Normal Transactions', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Fraud by hour of day
fraud_by_hour = df.groupby('hour_of_day')['fraud'].agg(['sum', 'count'])
fraud_by_hour['rate'] = fraud_by_hour['sum'] / fraud_by_hour['count'] * 100
axes[0].bar(fraud_by_hour.index, fraud_by_hour['rate'], color='#e74c3c', alpha=0.8, edgecolor='none')
axes[0].set_title('Fraud Rate by Hour of Day (%)', fontweight='bold')
axes[0].set_xlabel('Hour of Day')
axes[0].set_ylabel('Fraud Rate (%)')
axes[0].set_xticks(range(0, 24, 2))

# Fraud by transaction type
fraud_by_type = df.groupby('transaction_type')['fraud'].mean() * 100
axes[1].barh(fraud_by_type.sort_values().index, fraud_by_type.sort_values().values,
              color='#e74c3c', alpha=0.8, edgecolor='none')
axes[1].set_title('Fraud Rate by Transaction Type (%)', fontweight='bold')
axes[1].set_xlabel('Fraud Rate (%)')

plt.suptitle('Fraud Patterns in Mobile Money Transactions', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Feature Engineering & Preprocessing

In [ ]:
le_type   = LabelEncoder()
le_region = LabelEncoder()
df['txn_type_enc']     = le_type.fit_transform(df['transaction_type'])
df['agent_region_enc'] = le_region.fit_transform(df['agent_region'])

# Engineer additional features
df['log_amount'] = np.log1p(df['amount_ksh'])
df['log_volume'] = np.log1p(df['prev_7day_volume_ksh'])
df['is_night']   = ((df['hour_of_day'] >= 22) | (df['hour_of_day'] <= 5)).astype(float)
df['amount_per_txn'] = df['prev_7day_volume_ksh'] / (df['prev_7day_txn_count'] + 1)

feature_cols = ['log_amount', 'hour_of_day', 'day_of_week', 'account_age_days',
                'prev_7day_txn_count', 'log_volume', 'recipient_age_days',
                'failed_pin_attempts', 'distance_from_home_km', 'is_new_recipient',
                'txn_type_enc', 'agent_region_enc', 'is_night', 'amount_per_txn']

X = df[feature_cols].values
y = df['fraud'].values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f"Feature matrix: {X_scaled.shape}")
print(f"Fraud rate: {y.mean()*100:.2f}%")
print(f"Features used: {feature_cols}")

## 4. Anomaly Detection Models

We apply three unsupervised anomaly detection algorithms:
- **Isolation Forest:** Isolates anomalies using random partitioning trees
- **Local Outlier Factor (LOF):** Compares local density of points to their neighbors
- **One-Class SVM:** Learns a decision boundary around normal transactions

In [ ]:
iso_forest = IsolationForest(contamination=0.05, random_state=42, n_estimators=200, n_jobs=-1)
iso_forest.fit(X_scaled)

iso_pred = iso_forest.predict(X_scaled)
iso_pred_binary = (iso_pred == -1).astype(int)
iso_scores = -iso_forest.score_samples(X_scaled)

print("Isolation Forest:")
print(f"  Flagged as anomaly: {iso_pred_binary.sum()} ({iso_pred_binary.mean()*100:.1f}%)")
print(f"  True fraud rate among flagged: {y[iso_pred_binary == 1].mean()*100:.1f}%")
print(f"\nClassification Report:")
print(classification_report(y, iso_pred_binary, target_names=['Normal', 'Fraud']))

In [ ]:
lof = LocalOutlierFactor(n_neighbors=20, contamination=0.05, n_jobs=-1)
lof_pred = lof.fit_predict(X_scaled)
lof_pred_binary = (lof_pred == -1).astype(int)
lof_scores = -lof.negative_outlier_factor_

print("Local Outlier Factor:")
print(f"  Flagged as anomaly: {lof_pred_binary.sum()} ({lof_pred_binary.mean()*100:.1f}%)")
print(f"  True fraud rate among flagged: {y[lof_pred_binary == 1].mean()*100:.1f}%")
print(f"\nClassification Report:")
print(classification_report(y, lof_pred_binary, target_names=['Normal', 'Fraud']))

In [ ]:
# One-Class SVM (train only on normal transactions)
X_normal = X_scaled[y == 0]
ocsvm = OneClassSVM(kernel='rbf', nu=0.05, gamma='scale')
ocsvm.fit(X_normal)

ocsvm_pred = ocsvm.predict(X_scaled)
ocsvm_pred_binary = (ocsvm_pred == -1).astype(int)
ocsvm_scores = -ocsvm.score_samples(X_scaled)

print("One-Class SVM:")
print(f"  Flagged as anomaly: {ocsvm_pred_binary.sum()} ({ocsvm_pred_binary.mean()*100:.1f}%)")
print(f"  True fraud rate among flagged: {y[ocsvm_pred_binary == 1].mean()*100:.1f}%")
print(f"\nClassification Report:")
print(classification_report(y, ocsvm_pred_binary, target_names=['Normal', 'Fraud']))

In [ ]:
models_comparison = {
    'Isolation Forest': {'predictions': iso_pred_binary, 'scores': iso_scores},
    'LOF': {'predictions': lof_pred_binary, 'scores': lof_scores},
    'One-Class SVM': {'predictions': ocsvm_pred_binary, 'scores': ocsvm_scores},
}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors_roc = ['#3498db', '#2ecc71', '#e74c3c']
for (name, m), color in zip(models_comparison.items(), colors_roc):
    fpr, tpr, _ = roc_curve(y, m['scores'])
    auc_val = roc_auc_score(y, m['scores'])
    axes[0].plot(fpr, tpr, lw=2, color=color, label=f'{name} (AUC={auc_val:.3f})')

axes[0].plot([0,1],[0,1], 'k--', lw=1)
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curves — Anomaly Detection', fontweight='bold')
axes[0].legend()

for (name, m), color in zip(models_comparison.items(), colors_roc):
    precision, recall, _ = precision_recall_curve(y, m['scores'])
    ap = average_precision_score(y, m['scores'])
    axes[1].plot(recall, precision, lw=2, color=color, label=f'{name} (AP={ap:.3f})')

axes[1].axhline(y=y.mean(), color='gray', ls='--', label=f'Baseline ({y.mean():.3f})')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall Curves', fontweight='bold')
axes[1].legend()

plt.suptitle('Anomaly Detection Model Comparison', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
pca = PCA(n_components=2)
X_2d = pca.fit_transform(X_scaled)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# True labels
for cls, label, color in [(0, 'Normal', 'steelblue'), (1, 'Fraud', 'red')]:
    mask = y == cls
    axes[0].scatter(X_2d[mask, 0], X_2d[mask, 1], c=color, label=label, alpha=0.4, s=8)
axes[0].set_title('True Labels (PCA 2D)', fontweight='bold')
axes[0].legend()

# Isolation Forest predictions
for cls, label, color in [(0, 'Normal', 'steelblue'), (1, 'Flagged', 'red')]:
    mask = iso_pred_binary == cls
    axes[1].scatter(X_2d[mask, 0], X_2d[mask, 1], c=color, label=label, alpha=0.4, s=8)
axes[1].set_title('Isolation Forest Predictions', fontweight='bold')
axes[1].legend()

# LOF predictions
for cls, label, color in [(0, 'Normal', 'steelblue'), (1, 'Flagged', 'red')]:
    mask = lof_pred_binary == cls
    axes[2].scatter(X_2d[mask, 0], X_2d[mask, 1], c=color, label=label, alpha=0.4, s=8)
axes[2].set_title('LOF Predictions', fontweight='bold')
axes[2].legend()

for ax in axes:
    ax.set_xlabel('PC1')
    ax.set_ylabel('PC2')

plt.suptitle('Anomaly Detection Visualization (PCA 2D)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(14, 5))
plt.subplot(1, 2, 1)
plt.hist(iso_scores[y == 0], bins=50, alpha=0.6, color='steelblue', label='Normal', density=True)
plt.hist(iso_scores[y == 1], bins=30, alpha=0.7, color='red', label='Fraud', density=True)
plt.xlabel('Isolation Forest Anomaly Score')
plt.ylabel('Density')
plt.title('Anomaly Score Distribution', fontweight='bold')
plt.legend()

plt.subplot(1, 2, 2)
cm = confusion_matrix(y, iso_pred_binary)
sns.heatmap(cm, annot=True, fmt='d', cmap='Oranges',
            xticklabels=['Normal', 'Fraud'], yticklabels=['Normal', 'Fraud'])
plt.title('Confusion Matrix — Isolation Forest', fontweight='bold')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.tight_layout()
plt.show()

## 5. Threshold Analysis

The contamination parameter controls the anomaly threshold. We analyze how different thresholds affect precision and recall — critical for business decisions about false positive costs.

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

thresholds = np.percentile(iso_scores, np.arange(80, 99, 1))
precisions, recalls, f1s, flagged_rates = [], [], [], []

for thresh in thresholds:
    preds = (iso_scores >= thresh).astype(int)
    precisions.append(precision_score(y, preds, zero_division=0))
    recalls.append(recall_score(y, preds, zero_division=0))
    f1s.append(f1_score(y, preds, zero_division=0))
    flagged_rates.append(preds.mean() * 100)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(flagged_rates, precisions, 'b-o', label='Precision', lw=2, markersize=5)
axes[0].plot(flagged_rates, recalls,    'r-o', label='Recall',    lw=2, markersize=5)
axes[0].plot(flagged_rates, f1s,        'g-o', label='F1 Score',  lw=2, markersize=5)
axes[0].set_xlabel('Flagged Rate (%)')
axes[0].set_ylabel('Score')
axes[0].set_title('Precision / Recall / F1 vs Flagged Rate', fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Business cost analysis: false negatives (missed fraud) vs false positives (wrongly flagged)
COST_FN = 5000   # Cost of missing a fraud (KSh) — avg fraud loss
COST_FP = 50     # Cost of false alert (manual review cost)

costs = []
for thresh in thresholds:
    preds = (iso_scores >= thresh).astype(int)
    fn = ((preds == 0) & (y == 1)).sum()
    fp = ((preds == 1) & (y == 0)).sum()
    costs.append(fn * COST_FN + fp * COST_FP)

axes[1].plot(flagged_rates, costs, 'purple', lw=2, marker='o', markersize=5)
best_idx = np.argmin(costs)
axes[1].axvline(flagged_rates[best_idx], color='red', ls='--', lw=2, label=f'Optimal: {flagged_rates[best_idx]:.1f}%')
axes[1].set_xlabel('Flagged Rate (%)')
axes[1].set_ylabel('Total Business Cost (KSh)')
axes[1].set_title('Business Cost vs Flagged Rate', fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('Isolation Forest Threshold Analysis', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"Optimal flagging threshold: flag {flagged_rates[best_idx]:.1f}% of transactions")
print(f"Minimum business cost: KSh {costs[best_idx]:,.0f}")

## 6. Feature Importance via Anomaly Score Correlation

In [ ]:
# Compute correlation between each feature and anomaly score
feature_correlations = []
for i, feat in enumerate(feature_cols):
    corr = np.corrcoef(X_scaled[:, i], iso_scores)[0, 1]
    feature_correlations.append((feat, abs(corr), corr))

feat_corr_df = pd.DataFrame(feature_correlations, columns=['Feature', 'AbsCorr', 'Corr'])
feat_corr_df = feat_corr_df.sort_values('AbsCorr', ascending=False)

plt.figure(figsize=(10, 6))
colors_bar = ['#e74c3c' if c > 0 else '#3498db' for c in feat_corr_df['Corr']]
plt.barh(feat_corr_df['Feature'][::-1], feat_corr_df['Corr'][::-1], color=colors_bar[::-1], alpha=0.85)
plt.axvline(0, color='black', lw=0.8)
plt.xlabel('Correlation with Anomaly Score')
plt.title('Feature Correlation with Isolation Forest Anomaly Score', fontweight='bold')
plt.tight_layout()
plt.show()

print("Top features most correlated with anomaly scores:")
print(feat_corr_df[['Feature', 'AbsCorr']].head(8).to_string(index=False))

## 7. Conclusions

### Key Findings

1. **Isolation Forest** achieves the best trade-off between precision and recall for mobile money fraud detection, benefiting from its tree-based isolation mechanism that handles high-dimensional feature spaces efficiently.

2. **Key fraud indicators** (by feature correlation with anomaly score):
   - **Transaction amount** — fraud transactions are significantly larger
   - **Account age** — fraudulent accounts tend to be newer
   - **7-day transaction frequency** — burst activity precedes fraud events
   - **Recipient age** — fraud goes to newly registered recipients
   - **Distance from home** — transactions far from usual location are suspicious

3. **Business Cost Analysis:** The optimal threshold balances the cost of missed fraud (~KSh 5,000 per incident) against the cost of false alerts (~KSh 50 for manual review).

### Real-World Deployment Considerations
- **Real-time scoring:** Isolation Forest predictions must complete in <100ms per transaction
- **Model drift:** Fraud patterns evolve; models need monthly retraining
- **Regulatory compliance:** CBK (Kenya) and BOU (Uganda) require explainable AI for financial decisions
- **False positive impact:** Blocking legitimate transactions harms user trust in M-Pesa/MTN systems

### Next Steps
- **Autoencoder-based detection:** Deep learning reconstruction error as anomaly signal
- **Graph Neural Networks:** Model transaction networks to detect money laundering rings
- **Supervised approach:** Use labeled fraud data for XGBoost classification with SMOTE oversampling
- **Federated learning:** Train across telecoms without sharing raw transaction data